In [79]:
import pandas as pd
import numpy as np
import sklearn
import seaborn as sns
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

data = pd.read_csv("Suppy_Chain_Shipment_Data.csv")

In [80]:
for idx in range(len(data['weight (kilograms)'])):
    if data['weight (kilograms)'][idx][0] == "S":
        text = data['weight (kilograms)'][idx].split()
        asn_dn, id = text[1], text[2].strip("()").split(":")[1]

        weight = data[(data['id'] == int(id)) & (data['asn/dn #'] == asn_dn)]['weight (kilograms)'].values[0]
        data.loc[idx, 'weight (kilograms)'] = weight

for idx in range(len(data['freight cost (usd)'])):
    if data['freight cost (usd)'][idx][0] == "S":
        text = data['freight cost (usd)'][idx].split()
        asn_dn, id = text[1], text[2].strip("()").split(":")[1]

        freight_cost = data[(data['id'] == int(id)) & (data['asn/dn #'] == asn_dn)]['freight cost (usd)'].values[0]
        data.loc[idx, 'freight cost (usd)'] = freight_cost

In [81]:
for col in ["scheduled delivery date", "delivered to client date", "delivery recorded date"]:
    data[col] = pd.to_datetime(data[col], format="%d-%b-%y", errors="coerce")

data[["scheduled delivery date", "delivered to client date", "delivery recorded date"]].dtypes

scheduled delivery date     datetime64[us]
delivered to client date    datetime64[us]
delivery recorded date      datetime64[us]
dtype: object

In [82]:
date_cols = [
    "scheduled delivery date",
    "delivered to client date",
    "delivery recorded date"
]

for col in date_cols:
    data[col] = pd.to_datetime(data[col], errors="coerce")

data["delivery_status"] = np.select(
    [
        data["delivered to client date"] < data["scheduled delivery date"],
        data["delivered to client date"] == data["scheduled delivery date"],
        data["delivered to client date"] > data["scheduled delivery date"]
    ],
    [
        "Early",
        "On Time",
        "Late"
    ],
    default = "Unknown"
)

data["delivery_delay_days"] = (
    data["delivered to client date"]
    - data["scheduled delivery date"]
).dt.days

In [83]:
date_cols = [
    "pq first sent to client date",
    "po sent to vendor date",
    "scheduled delivery date",
    "delivered to client date",
    "delivery recorded date"
]

for col in date_cols:
    data[col] = pd.to_datetime(
        data[col],
        errors="coerce"
    )

/var/folders/w7/g9bqsf9s4gn4s8vj29jr8wqw0000gn/T/ipykernel_19205/3844352272.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data[col] = pd.to_datetime(
/var/folders/w7/g9bqsf9s4gn4s8vj29jr8wqw0000gn/T/ipykernel_19205/3844352272.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data[col] = pd.to_datetime(


In [84]:
financial_cols = [
    "unit price",
    "pack price",
    "line item value",
    "freight cost (usd)",
    "line item insurance (usd)"
]

data["freight_status"] = np.select(
    [
        data["freight cost (usd)"].eq(
            "Freight Included in Commodity Cost"
        ),
        data["freight cost (usd)"].eq(
            "Invoiced Separately"
        )
    ],
    [
        "Included in Commodity Cost",
        "Invoiced Separately"
    ],
    default=pd.NA
)

for col in financial_cols:
    data[col] = pd.to_numeric(
        data[col],
        errors="coerce"
    )

In [85]:
data['weight (kilograms)'] = pd.to_numeric(
    data['weight (kilograms)'],
    errors="coerce"
)

In [86]:
data['weight (kilograms)'] = data['weight (kilograms)'].replace(np.nan, data['weight (kilograms)'].median())
data['freight cost (usd)'] = data['freight cost (usd)'].replace(np.nan, data['freight cost (usd)'].median())

In [87]:
data["freight_per_kg"] = data["freight cost (usd)"] / data["weight (kilograms)"]
data["freight_per_unit"] = data["freight cost (usd)"] / data["line item quantity"]
data["freight_pct_of_value"] = (
    100 * data["freight cost (usd)"] / data["line item value"]
)
data["delivery_year"] = data["delivered to client date"].dt.year

# Guard against impossible/infinite ratios caused by zero or missing denominators.
ratio_cols = ["freight_per_kg", "freight_per_unit", "freight_pct_of_value"]
data[ratio_cols] = data[ratio_cols].replace([np.inf, -np.inf], np.nan)


In [88]:
data.to_csv("Supply_Chain_Shipment_Data_Cleaned.csv", index=False)

In [91]:
data.drop(columns = 'freight_status').isna().sum().sum()

np.int64(10814)